# 02 - Single Asset (ETH only)

Deep dive on ETH without any other market.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Fetch

In [2]:
eth_1d = fetch("ETHUSDT", "1d", 500)
print("rows:", len(eth_1d), " last:", eth_1d.index[-1].date())

rows: 499  last: 2026-10-03


## 2. Load from DB (6 years)

In [3]:
hist = sql("SELECT open_time, close FROM ohlcv WHERE symbol='ETHUSDT' AND timeframe='1d' ORDER BY open_time")
hist["date"] = pd.to_datetime(hist.open_time, unit="ms")
hist = hist.set_index("date")["close"]
print("range:", hist.index.min().date(), "->", hist.index.max().date(), "n=", len(hist))

range: 2020-10-05 -> 2026-10-02 n= 2189


## 3. Rolling stats

In [4]:
r = hist.pct_change().dropna()
print("mean daily:", f"{r.mean()*100:+.4f}%")
print("std daily :", f"{r.std()*100:.4f}%")
print("annualized vol:", f"{r.std()*np.sqrt(365)*100:.2f}%")
print("sharpe:", f"{r.mean()/r.std()*np.sqrt(365):.2f}")

mean daily: +0.1737%
std daily : 4.0377%
annualized vol: 77.14%
sharpe: 0.82


## 4. Levels — support / resistance

In [5]:
recent = eth_1d.tail(60)
support = recent["low"].min()
resistance = recent["high"].max()
print(f"60d support    : {support:.2f}")
print(f"60d resistance : {resistance:.2f}")
print(f"current        : {float(eth_1d.close.iloc[-1]):.2f}")
print(f"range position : {(eth_1d.close.iloc[-1] - support)/(resistance - support)*100:.1f}%")

60d support    : 1853.62
60d resistance : 2807.34
current        : 2687.23
range position : 87.4%


## 5. Action

In [6]:
cur = float(eth_1d.close.iloc[-1])
pos = (cur - support) / (resistance - support)

if pos < 0.2:
    action, reason = "LONG", "near 60d support — bounce zone"
elif pos > 0.8:
    action, reason = "SHORT", "near 60d resistance — fade zone"
else:
    action, reason = "FLAT", "mid-range — no edge"

action_block("ETH SINGLE ASSET", action, reason,
    price=cur, extra={"range_pos": f"{pos*100:.1f}%"})

  ETH SINGLE ASSET
  time   : 2026-10-04 10:39 UTC
  price  : 2,687.23
  ACTION : SHORT
  REASON : near 60d resistance — fade zone
  range_pos : 87.4%


{'action': 'SHORT',
 'reason': 'near 60d resistance — fade zone',
 'price': 2687.23,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': None}